# Final pipeline (coordinated by Member 4)
Loads saved checkpoints only - **do not retrain here**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, yaml
REPO = "/content/nighttime-flare-removal"
if not os.path.exists(REPO):
    !git clone https://github.com/USERNAME/nighttime-flare-removal.git {REPO}   # <-- replace USERNAME
%cd {REPO}
!git pull
!pip -q install pyyaml scikit-image tqdm
sys.path.insert(0, REPO)
cfg = yaml.safe_load(open("configs/default.yaml"))
import torch
print("CUDA:", torch.cuda.is_available())

In [ ]:
from src.evaluation.evaluate import load_checkpoint, compare, save_comparison_grid
from src.data.dataset import PairedFolderDataset
from torch.utils.data import DataLoader
PROJECT = cfg["project"]["drive_root"]; dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_ds = PairedFolderDataset("/content/flare_test/input", "/content/flare_test/target", crop_size=256, train=False)  # set paths
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
ck = PROJECT + "/checkpoints"
models = {"Plain U-Net": load_checkpoint("unet", ck + "/baseline/unet_best.pth", cfg, dev),
          "Attention + multiscale": load_checkpoint("attention_multiscale", ck + "/improved/attention_multiscale_best.pth", cfg, dev)}

In [ ]:
table = compare(models, test_loader, dev, out_json=PROJECT + "/results/metrics/final_metrics.json")
save_comparison_grid(models, test_loader, dev, PROJECT + "/results/figures/final_grid.png", n=4)
from IPython.display import Image, display; display(Image(PROJECT + "/results/figures/final_grid.png"))

## Demo: restore a new uploaded image
(Estimated restoration - details fully lost to saturation cannot be recovered.)

In [ ]:
from google.colab import files
from PIL import Image as PILImage
from src.inference.predict import restore_image
up = files.upload(); path = next(iter(up))
src_img = PILImage.open(path)
out = restore_image(models["Attention + multiscale"], src_img, dev)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 6)); ax[0].imshow(src_img); ax[0].set_title("input"); ax[1].imshow(out); ax[1].set_title("restored")
for a in ax: a.axis("off")
plt.show(); out.save(PROJECT + "/results/figures/demo_output.png")